# H&M 2년 M1 고정 CLV·N/V 구간별 신규상품 오차 진단

기존 H&M 2년 seed 42 M1 체크포인트를 **재학습하지 않고** 분석합니다.

- 구매기회: 같은 고객이 같은 날짜에 구매한 상품들을 한 번의 구매기회로 묶음
- N: 학습기간 구매기회 수
- V: 구매기회당 평균 구매금액
- 고정 historical CLV proxy: `N × V`(학습기간 관측 구매금액)
- 평가: 기존 H&M 2년 validation의 신규상품 정답만 사용하며 test·holdout은 열지 않음
- 비교: 저·중·고CLV, 전체 고객 N/V 4유형, 고CLV 내부 N/V 구성 3유형
- 상품 비교: 상품군, 가격 백분위, 학습 구매고객 수, 정확히 같은 상품의 반복구매 비율, 고객 구매이력 상품군 및 임베딩 유사도

H&M에서는 동일 상품 반복구매가 희소하므로 반복구매 비율은 보조 진단으로만 해석합니다. 이 결과는 단일 시드의 기술적 진단이며 모형 선택이나 통계적 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '46f485bcb266cb796aa170cf52290c7ebcc8bc5b'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA

import subprocess
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
print('진단 코드 고정:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_fixed_segment_error_diagnostic_hm2y import (
    configure_hm2y_fixed_segment_error_diagnostic,
    preflight_summary,
    run_hm2y_fixed_segment_error_diagnostic,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_hm2y_fixed_segment_error_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_hm_m1_fixed_clv_nv_segment_error_diagnostic_hm2y_v1'
    ),
    m1_checkpoint_dir='/content/drive/MyDrive/논문/data/results_v3_hm',
    eval_batch_size=256,
    top_examples=20,
)
summary = preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['split'] == 'existing_hm2y_validation'
assert summary['test_executed'] is False
assert summary['holdout_executed'] is False
assert summary['new_item_task'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
report = run_hm2y_fixed_segment_error_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) 저·중·고CLV 사용자 수와 M1 성과')
display(report['segment_population'])
display(report['segment_metrics'])

print('2) 전체 고객 N/V 4유형과 M1 성과')
display(report['nv_quadrant_population'])
display(report['nv_quadrant_metrics'])
display(report['nv_quadrant_contrasts'])

print('3) 고CLV 내부 N/V 구성 3유형과 M1 성과')
display(report['high_clv_composition_population'])
display(report['high_clv_composition_metrics'])
display(report['high_clv_composition_contrasts'])

print('4) 정답 누락상품과 Top-10 오추천상품의 특성 차이')
display(report['contrasts'])

print('5) H&M 상품군별 누락·오추천 및 실제 상품 예시')
display(report['category_summary'])
display(report['examples'])
print('저장 파일:', json.dumps(report['paths'], ensure_ascii=False, indent=2))